# 🏟️ Évaluation du Pipeline Amélioré — Football CV (sans entraînement)

**PFE INSEA 2025-2026 — Elkhalil DAHANI**

Ce notebook **n'entraîne aucun modèle** : il charge les **2 modèles déjà entraînés** (Modèle 1 *baseline* SoccerNet, Modèle 2 *enrichi* SoccerNet+Roboflow) et la **data** depuis les *inputs Kaggle*, puis déroule tous les **sprints d'évaluation** du pipeline **avec les améliorations** apportées au code :

| Sprint | Module | Amélioration évaluée | Métrique |
|---|---|---|---|
| 1 | Détection | Comparaison **M1 vs M2** | mAP@50, mAP@50-95, par classe |
| 2 | Tracking | **Interpolation** + ID switches | recall, sauts d'ID |
| 3 | Équipes | KMeans HSV + **vote temporel** | silhouette, stabilité |
| 4 | Homographie | **Statique vs DYNAMIQUE** (flux optique) | dérive de reprojection |
| 5 | Tactique | **PPDA v2 par événement** + score continu | événements vs frames |
| 6 | Validation | Calibration vs **StatsBomb** (Euro 2020) | erreur abs. |
| 7 | Synthèse | Récapitulatif | tableau + JSON |

> ⚠️ Renseigne les **slugs Kaggle** dans la cellule CONFIG ci-dessous. Une auto-détection est fournie en secours.

## Sprint 0 — Configuration & installation

In [ ]:
# Dépendances (Kaggle a déjà torch/ultralytics, mais on fige les versions utiles)
import subprocess, sys
def pipq(*pkgs):
    subprocess.run([sys.executable,"-m","pip","install","-q",*pkgs], check=False)
pipq("ultralytics==8.4.40","opencv-python-headless","statsbombpy","mplsoccer")
print("✅ Dépendances installées")

In [ ]:
import os, sys, glob, json, warnings, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import cv2
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# 1) CODE SOURCE AMÉLIORÉ : cloné depuis GitHub (ou input Kaggle)
# ─────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/DAHANIElkhalil25/football_cv.git"
REPO_DIR  = "/kaggle/working/football_cv"
if not Path(REPO_DIR, "src").exists():
    # a) tenter un input Kaggle nommé "football_cv" / "football-cv"
    cand = [p for p in glob.glob("/kaggle/input/*") if "football" in p.lower() and Path(p,"src").exists()]
    if cand:
        REPO_DIR = cand[0]
    else:
        subprocess.run(["git","clone","-q",REPO_URL,REPO_DIR], check=False)
sys.path.insert(0, REPO_DIR)
print("📦 Code source :", REPO_DIR, "| src/ présent :", Path(REPO_DIR,"src").exists())

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2) SLUGS KAGGLE — ⚙️ À RENSEIGNER (sinon auto-détection)
# ─────────────────────────────────────────────────────────────
# Mets ici les chemins EXACTS de tes inputs Kaggle :
MODEL1_PATH   = ""   # ex: "/kaggle/input/yolov8m-soccernet-baseline/best.pt"
MODEL2_PATH   = ""   # ex: "/kaggle/input/yolov8m-soccernet-roboflow/best.pt"
DATA_YOLO_DIR = ""   # dossier YOLO converti : contient images/test + labels/test
HOMOGRAPHY_NPY= ""   # ex: "/kaggle/input/.../homography.npy" (optionnel)
TEST_SEQ_DIR  = ""   # dossier d'images d'une séquence test (frames .jpg) (optionnel)

def autodetect(patterns, must_contain_file=None, want_dir=False):
    for base in glob.glob("/kaggle/input/*"):
        for pat in patterns:
            for hit in glob.glob(os.path.join(base,"**",pat), recursive=True):
                if want_dir and Path(hit).is_dir(): return hit
                if not want_dir and Path(hit).is_file(): return hit
    return ""

if not MODEL1_PATH or not Path(MODEL1_PATH).exists():
    pts = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
    if len(pts) >= 1 and not MODEL1_PATH: MODEL1_PATH = pts[0]
    if len(pts) >= 2 and not MODEL2_PATH: MODEL2_PATH = pts[1]
if not DATA_YOLO_DIR:
    for d in glob.glob("/kaggle/input/**/images", recursive=True):
        if Path(d,"test").exists() or Path(d).parent.joinpath("labels").exists():
            DATA_YOLO_DIR = str(Path(d).parent); break
if not HOMOGRAPHY_NPY:
    HOMOGRAPHY_NPY = autodetect(["homography*.npy","*homography*.npy"])

print("M1            :", MODEL1_PATH)
print("M2            :", MODEL2_PATH)
print("DATA_YOLO_DIR :", DATA_YOLO_DIR)
print("HOMOGRAPHY    :", HOMOGRAPHY_NPY or "(aucune)")
print("TEST_SEQ_DIR  :", TEST_SEQ_DIR or "(à définir si besoin)")

In [ ]:
# Constantes & utilitaires graphiques
DEVICE = 0 if __import__('torch').cuda.is_available() else 'cpu'
IMGSZ  = 1280
CLASS_NAMES  = {0:"player",1:"goalkeeper",2:"referee",3:"ball"}
TEAM_COLORS  = {0:"#00CED1",1:"#FF4500",2:"#FFD700"}
PITCH_L, PITCH_W = 105.0, 68.0
RESULTS = {}
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110})

def draw_pitch(ax, L=PITCH_L, W=PITCH_W):
    ax.set_xlim(-3,L+3); ax.set_ylim(-3,W+3); ax.set_aspect("equal")
    ax.add_patch(plt.Rectangle((0,0),L,W,ec="#444",fc="#eaf3ea",lw=1.5))
    ax.plot([L/2,L/2],[0,W],color="#888",lw=1)
    ax.add_patch(plt.Circle((L/2,W/2),9.15,ec="#888",fc="none",lw=1))
    for x0 in (0,L-16.5):
        ax.add_patch(plt.Rectangle((x0,(W-40.32)/2),16.5,40.32,ec="#888",fc="none",lw=1))
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")

print("DEVICE =", DEVICE)

## Sprint 1 — Détection : comparaison Modèle 1 vs Modèle 2

On évalue les deux modèles sur le **même jeu de test 100 % SoccerNet** (stratégie de *fusion asymétrique* : Roboflow uniquement à l'entraînement). On compare mAP@50, mAP@50-95 et la performance **par classe** (le ballon étant le point faible).

In [ ]:
from ultralytics import YOLO
import yaml as _yaml

def build_test_yaml(data_dir):
    # Crée un data.yaml pointant vers le split test sur Kaggle.
    if not data_dir or not Path(data_dir).exists():
        return None
    y = {"path": str(data_dir),
         "train":"images/train","val":"images/val","test":"images/test",
         "nc":4, "names":{0:"player",1:"goalkeeper",2:"referee",3:"ball"}}
    out = "/kaggle/working/_eval_data.yaml"
    with open(out,"w") as f: _yaml.safe_dump(y,f,allow_unicode=True)
    return out

EVAL_YAML = build_test_yaml(DATA_YOLO_DIR)
print("YAML d'évaluation :", EVAL_YAML)

In [ ]:
def eval_model(path, name):
    if not path or not Path(path).exists() or not EVAL_YAML:
        print(f"⏭️  {name}: modèle ou data absent — sprint sauté"); return None
    m = YOLO(path)
    r = m.val(data=EVAL_YAML, split="test", imgsz=IMGSZ, device=DEVICE, verbose=False)
    per_class = {}
    try:
        for i, ap in zip(r.box.ap_class_index, r.box.maps):
            per_class[CLASS_NAMES.get(int(i),str(i))] = float(ap)
    except Exception:
        pass
    return {"map50":float(r.box.map50),"map5095":float(r.box.map),
            "precision":float(r.box.mp),"recall":float(r.box.mr),"per_class":per_class}

res_m1 = eval_model(MODEL1_PATH, "Modèle 1 (baseline)")
res_m2 = eval_model(MODEL2_PATH, "Modèle 2 (enrichi)")
RESULTS["detection"] = {"M1":res_m1,"M2":res_m2}
for n,r in [("M1",res_m1),("M2",res_m2)]:
    if r: print(f"{n}: mAP@50={r['map50']:.3f} | mAP@50-95={r['map5095']:.3f} | R={r['recall']:.3f}")

In [ ]:
# Visualisation comparative M1 vs M2
if res_m1 and res_m2:
    fig, axes = plt.subplots(1,2, figsize=(13,5))
    labels=["mAP@50","mAP@50-95","Precision","Recall"]
    m1=[res_m1["map50"],res_m1["map5095"],res_m1["precision"],res_m1["recall"]]
    m2=[res_m2["map50"],res_m2["map5095"],res_m2["precision"],res_m2["recall"]]
    x=np.arange(len(labels)); w=0.38
    axes[0].bar(x-w/2,m1,w,label="M1 baseline",color="#7aa6c2")
    axes[0].bar(x+w/2,m2,w,label="M2 enrichi",color="#e07a5f")
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels,rotation=15); axes[0].legend(); axes[0].set_title("Global")
    cls=sorted(set(res_m1["per_class"])|set(res_m2["per_class"]))
    if cls:
        xc=np.arange(len(cls))
        axes[1].bar(xc-w/2,[res_m1["per_class"].get(c,0) for c in cls],w,label="M1",color="#7aa6c2")
        axes[1].bar(xc+w/2,[res_m2["per_class"].get(c,0) for c in cls],w,label="M2",color="#e07a5f")
        axes[1].set_xticks(xc); axes[1].set_xticklabels(cls,rotation=15); axes[1].legend(); axes[1].set_title("mAP par classe")
    plt.tight_layout(); plt.show()
else:
    print("Comparaison non disponible (un modèle manquant).")

## Préparation commune — séquence de frames (sprints 2 à 5)

Les sprints suivants utilisent une **séquence d'images** (frames d'un extrait). On charge la séquence et on choisit le **Modèle 2** (meilleur) pour le pipeline tactique.

In [ ]:
PIPE_MODEL = MODEL2_PATH if (MODEL2_PATH and Path(MODEL2_PATH).exists()) else MODEL1_PATH

# Récupérer une séquence de frames
if not TEST_SEQ_DIR:
    cands = sorted(glob.glob("/kaggle/input/**/img1", recursive=True))  # format MOT
    if cands: TEST_SEQ_DIR = cands[0]
    else:
        anyimgs = sorted(glob.glob("/kaggle/input/**/test/**/*.jpg", recursive=True))
        if anyimgs: TEST_SEQ_DIR = str(Path(anyimgs[0]).parent)

N_FRAMES = 120
frame_paths = []
if TEST_SEQ_DIR and Path(TEST_SEQ_DIR).exists():
    frame_paths = sorted(glob.glob(os.path.join(TEST_SEQ_DIR,"*.jpg")))[:N_FRAMES] or \
                  sorted(glob.glob(os.path.join(TEST_SEQ_DIR,"*.png")))[:N_FRAMES]
print("Séquence :", TEST_SEQ_DIR, "|", len(frame_paths), "frames")
print("Modèle pipeline :", PIPE_MODEL)

## Sprint 2 — Tracking : interpolation des trajectoires & ID switches

On suit les joueurs (ByteTrack), puis on applique l'**interpolation linéaire** des trous courts (`interpolate_missing_tracks`). On mesure le **gain de détections** et une approximation des **changements d'identité** (`count_identity_switches`).

In [ ]:
all_tracks = []
RESULTS["tracking"] = {}
if frame_paths and PIPE_MODEL:
    from src.tracking.tracker import ByteTrackerWrapper, interpolate_missing_tracks, count_identity_switches
    BYTE_CFG = str(Path(REPO_DIR,"config","bytetrack.yaml"))
    tracker = ByteTrackerWrapper(model_path=PIPE_MODEL, bytetrack_cfg=BYTE_CFG, device=DEVICE, imgsz=IMGSZ)
    for i,fp in enumerate(frame_paths):
        frame = cv2.imread(fp)
        objs = tracker.update(frame)
        all_tracks.append({"frame_idx":i,"frame_path":fp,
            "tracks":[{"track_id":o.track_id,"bbox":list(o.bbox_xyxy),
                       "bbox_xyxy":list(o.bbox_xyxy),"class_id":o.class_id,
                       "conf":o.confidence} for o in objs]})
    n_before = sum(len(f["tracks"]) for f in all_tracks)
    sw_before = count_identity_switches(all_tracks)
    all_tracks = interpolate_missing_tracks(all_tracks, max_gap=5)
    n_after = sum(len(f["tracks"]) for f in all_tracks)
    RESULTS["tracking"] = {"detections_before":n_before,"detections_after":n_after,
                            "interpolated":n_after-n_before,"id_switch_est":sw_before}
    print(f"Détections: {n_before} → {n_after} (+{n_after-n_before} interpolées)")
    print("ID switches (estimation):", sw_before)
else:
    print("⏭️ Pas de séquence/modèle — sprint tracking sauté")

In [ ]:
if all_tracks:
    counts=[len(f["tracks"]) for f in all_tracks]
    real=[sum(1 for t in f["tracks"] if not t.get("interpolated")) for f in all_tracks]
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot(counts,label="Après interpolation",color="#e07a5f")
    ax.plot(real,label="Détections réelles",color="#7aa6c2",ls="--")
    ax.set_xlabel("Frame"); ax.set_ylabel("Nb objets suivis"); ax.legend()
    ax.set_title("Sprint 2 — Remplissage des trous par interpolation"); plt.show()

## Sprint 3 — Classification d'équipes : KMeans HSV + vote temporel

On entraîne le classifieur couleur (HSV, patch torse 20-60 %), on calcule le **coefficient de silhouette**, puis on mesure la **stabilité** apportée par le **vote temporel majoritaire** (anti-*flickering*).

In [ ]:
RESULTS["teams"]={}
team_map={}
if all_tracks:
    from src.team_classifier.kmeans_classifier import TeamColorKMeansClassifier
    from sklearn.metrics import silhouette_score
    clf = TeamColorKMeansClassifier(n_clusters=3)
    samples = clf.collect_mean_colors(all_tracks, cv2.imread)
    if len(samples)>=10:
        clf.fit_from_samples(samples)
        labels = clf.model.predict(samples)
        sil = float(silhouette_score(samples, labels)) if len(set(labels))>1 else float("nan")
        # Stabilité: % de track_id dont l'équipe NE change PAS frame à frame
        raw=defaultdict(list)
        for ft in all_tracks:
            frame=cv2.imread(ft["frame_path"])
            for a in clf.assign_tracks(frame, ft["tracks"]):
                raw[a.track_id].append(a.team_id)
        flips=sum(any(v[i]!=v[i-1] for i in range(1,len(v))) for v in raw.values() if len(v)>1)
        team_map = clf.build_temporal_voting(all_tracks, cv2.imread)  # stable
        n_tracks=len([v for v in raw.values() if len(v)>1])
        stability = 1.0 - (flips/max(1,n_tracks))
        RESULTS["teams"]={"silhouette":sil,"stability_before_voting":stability,
                           "stability_after_voting":1.0,"n_samples":int(len(samples))}
        print(f"Silhouette={sil:.3f} | stabilité avant vote={stability:.2%} | après vote=100%")
    else:
        print("⏭️ Trop peu d'échantillons couleur")
else:
    print("⏭️ Sprint équipes sauté")

## Sprint 4 — Homographie : statique vs **dynamique** (flux optique)

Cœur de l'amélioration : on compare la **H statique** (calculée une fois) à la **H dynamique** (`DynamicHomography`, suivi de points par flux optique de Lucas-Kanade). On quantifie la **correction de dérive** appliquée par le suivi sur des repères du terrain. Une H statique laisserait les repères « glisser » lors des mouvements de caméra.

In [ ]:
RESULTS["homography"]={}
H_static=None
H_dyn_seq=[]
if HOMOGRAPHY_NPY and Path(HOMOGRAPHY_NPY).exists() and frame_paths:
    from src.geometry.dynamic_calibration import DynamicHomography, reprojection_drift, detect_shot_boundaries
    H_static = np.load(HOMOGRAPHY_NPY).astype(np.float64)
    frames_bgr=[cv2.imread(p) for p in frame_paths]
    cuts = detect_shot_boundaries(frames_bgr, similarity_threshold=0.6)
    dh = DynamicHomography(H_static, frames_bgr[0])
    H_dyn_seq=[H_static.copy()]
    for fr in frames_bgr[1:]:
        Ht = dh.update(fr)
        H_dyn_seq.append(Ht if Ht is not None else H_dyn_seq[-1])
    # Repères terrain fixes (mètres) — coins, centre, ronds de surface
    landmarks=np.array([[52.5,34.0],[0,0],[105,0],[0,68],[105,68],[16.5,34],[88.5,34]],dtype=np.float32)
    drift = reprojection_drift(landmarks, H_dyn_seq, H_static)
    RESULTS["homography"]={"shot_cuts":cuts,"reanchors":dh.state.reanchor_count,**drift}
    print("Coupures de plan détectées :", cuts)
    print(f"Correction dynamique moyenne: {drift['mean_correction_px']:.1f}px | max {drift['max_correction_px']:.1f}px")
else:
    print("⏭️ Homographie statique (.npy) ou séquence absente — sprint sauté")

In [ ]:
# Visualiser la trajectoire image d'un repère terrain : statique (fixe) vs dynamique (corrigé)
if H_dyn_seq and H_static is not None:
    pt=np.array([[[52.5,34.0]]],dtype=np.float32)  # rond central (terrain)
    stat=[cv2.perspectiveTransform(pt,np.linalg.inv(H_static))[0,0] for _ in H_dyn_seq]
    dyn =[cv2.perspectiveTransform(pt,np.linalg.inv(H))[0,0] for H in H_dyn_seq]
    stat=np.array(stat); dyn=np.array(dyn)
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot(dyn[:,0],label="x image — H dynamique (suit la caméra)",color="#e07a5f")
    ax.plot(stat[:,0],label="x image — H statique (figée)",color="#7aa6c2",ls="--")
    ax.set_xlabel("Frame"); ax.set_ylabel("Position image du rond central (px)")
    ax.set_title("Sprint 4 — La H dynamique suit le mouvement de caméra"); ax.legend(); plt.show()

## Sprint 5 — Tactique : **PPDA v2 par événement** vs v1 par frame

On projette les joueurs en BEV (via la **H dynamique** si dispo, sinon statique), on assigne les équipes (vote temporel), puis on compare :
- **v1** : actions défensives comptées **par frame** (surévaluation) ;
- **v2** : **événements** de pression (debounce) + restriction de **zone** + **adversaires uniquement** ;
et on calcule la **compacité** et le **contrôle Voronoï clippé**.

In [ ]:
RESULTS["tactics"]={}
def H_at(i):
    if H_dyn_seq: return H_dyn_seq[min(i,len(H_dyn_seq)-1)]
    return H_static
def to_bev(x,y,H):
    d=cv2.perspectiveTransform(np.array([[[x,y]]],dtype=np.float32),H); return float(d[0,0,0]),float(d[0,0,1])

frames_tac=[]
if all_tracks and (H_static is not None) and team_map:
    for ft in all_tracks:
        H=H_at(ft["frame_idx"])
        pxy=[]; teams=[]; ball=None
        for t in ft["tracks"]:
            bb=t["bbox_xyxy"]; fx=(bb[0]+bb[2])/2; fy=bb[3]
            bx,by=to_bev(fx,fy,H)
            if not(-10<=bx<=115 and -10<=by<=78): continue
            if t["class_id"]==3:
                ball=(bx,by)
            elif t["class_id"]==0:
                tm=team_map.get(t["track_id"])
                if tm in (0,1): pxy.append([bx,by]); teams.append(tm)
        frames_tac.append({"players_xy":np.array(pxy,dtype=float) if pxy else np.empty((0,2)),
                           "teams":np.array(teams),"ball_xy":ball})
    print(f"{len(frames_tac)} frames tactiques préparées | frames avec ballon: {sum(f['ball_xy'] is not None for f in frames_tac)}")
else:
    print("⏭️ Données BEV/équipes insuffisantes — sprint tactique limité")

In [ ]:
# PPDA v1 (par frame) vs v2 (par événement)
if frames_tac:
    from src.tactics.pressing import (estimate_defensive_actions_from_tracking,
                                       count_pressing_events, ppda_v2, calibrate_pressure_radius)
    # v1 : compter par frame (tous défenseurs vs ballon)
    v1=0
    for f in frames_tac:
        if f["ball_xy"] is None or len(f["players_xy"])==0: continue
        v1+=estimate_defensive_actions_from_tracking(f["players_xy"], np.array([f["ball_xy"]]), 4.0)
    # v2 : événements
    ev=count_pressing_events(frames_tac, pressure_radius_m=4.0, cooldown_frames=5, restrict_zone=True)
    RESULTS["tactics"]["ppda"]={"v1_actions_per_frame":int(v1),"v2":ev}
    print(f"v1 (par frame) : {v1} 'actions'  ➜  v2 (événements) : {ev['pressing_events']} "
          f"(frames sous pression={ev['frames_under_pressure']})")
    # courbe de calibration du rayon (sans cible: juste la sensibilité)
    cal=calibrate_pressure_radius(frames_tac, target_actions=ev['pressing_events'])
    xs=sorted(cal['curve']); ys=[cal['curve'][r] for r in xs]
    plt.figure(figsize=(8,4)); plt.plot(xs,ys,"o-",color="#e07a5f")
    plt.xlabel("Rayon de pression (m)"); plt.ylabel("Événements détectés")
    plt.title("Sprint 5 — Sensibilité du PPDA au rayon (à calibrer sur StatsBomb)"); plt.show()

In [ ]:
# Compacité + Voronoï clippé sur une frame représentative
if frames_tac:
    from src.tactics.compactness import compactness_metrics
    from src.tactics.voronoi import compute_control_map_clipped
    mid=len(frames_tac)//2
    f=frames_tac[mid]
    A=f["players_xy"][f["teams"]==0]; B=f["players_xy"][f["teams"]==1]
    if len(A)>=3 and len(B)>=3:
        cA=compactness_metrics(A); cB=compactness_metrics(B)
        ctrl=compute_control_map_clipped(A,B)
        RESULTS["tactics"]["compactness"]={"teamA":cA,"teamB":cB}
        RESULTS["tactics"]["voronoi"]={"A":ctrl["team_a_ratio"],"B":ctrl["team_b_ratio"]}
        fig,ax=plt.subplots(figsize=(10,6)); draw_pitch(ax)
        ax.scatter(A[:,0],A[:,1],c=TEAM_COLORS[0],s=90,ec="k",label="Équipe A",zorder=5)
        ax.scatter(B[:,0],B[:,1],c=TEAM_COLORS[1],s=90,ec="k",label="Équipe B",zorder=5)
        ax.set_title(f"Voronoï clippé — A={ctrl['team_a_ratio']:.0%} / B={ctrl['team_b_ratio']:.0%}")
        ax.legend(); plt.show()
        print("Compacité (aire enveloppe) A=%.0f m² | B=%.0f m²"%(cA['convex_hull_area'],cB['convex_hull_area']))

## Sprint 6 — Validation contre **StatsBomb** (Euro 2020)

On récupère les **événements de pression** d'un match Euro 2020 (données ouvertes StatsBomb), et on s'en sert de **cible** pour **calibrer le rayon** du PPDA v2. Cela transforme le seuil arbitraire (4 m) en paramètre **ajusté**. *(Nécessite l'accès Internet activé sur le notebook Kaggle.)*

In [ ]:
RESULTS["statsbomb"]={}
try:
    from statsbombpy import sb
    comps = sb.competitions()
    euro = comps[(comps.competition_name.str.contains("Euro",case=False,na=False))]
    if len(euro):
        cid=int(euro.iloc[0].competition_id); sid=int(euro.iloc[0].season_id)
        matches=sb.matches(competition_id=cid, season_id=sid)
        mid=int(matches.iloc[0].match_id)
        ev=sb.events(match_id=mid)
        n_press=int((ev.type=="Pressure").sum())
        teamsb=ev[ev.type=="Pressure"].team.value_counts().to_dict()
        RESULTS["statsbomb"]={"match_id":mid,"pressure_events_total":n_press,"by_team":teamsb}
        print(f"Match {matches.iloc[0]['home_team']} vs {matches.iloc[0]['away_team']} — "
              f"{n_press} events 'Pressure' StatsBomb")
        # calibrer notre rayon pour s'approcher (échelle ramenée à notre extrait)
        if RESULTS.get("tactics",{}).get("ppda"):
            # cible mise à l'échelle de la durée de notre extrait (~N_FRAMES/25 s vs 90 min)
            scale=(len(frames_tac)/25.0)/(95*60) if frames_tac else 0
            target=max(1,int(round(n_press*scale)))
            from src.tactics.pressing import calibrate_pressure_radius
            cal=calibrate_pressure_radius(frames_tac, target_actions=target)
            RESULTS["statsbomb"]["calibration"]={"scaled_target":target,**{k:cal[k] for k in ("best_radius","best_events","abs_error")}}
            print(f"Cible mise à l'échelle ≈ {target} events ➜ rayon calibré = {cal['best_radius']} m "
                  f"(events={cal['best_events']}, erreur={cal['abs_error']})")
    else:
        print("Compétition Euro introuvable dans StatsBomb open data")
except Exception as e:
    print("⏭️ StatsBomb indisponible (Internet désactivé ?):", e)

## Sprint 7 — Synthèse & export des résultats

In [ ]:
print(json.dumps(RESULTS, indent=2, ensure_ascii=False, default=str))
with open("/kaggle/working/evaluation_results.json","w") as f:
    json.dump(RESULTS,f,indent=2,ensure_ascii=False,default=str)
print("\n💾 Résultats sauvegardés → /kaggle/working/evaluation_results.json")

In [ ]:
# Tableau récapitulatif lisible
rows=[]
d=RESULTS.get("detection",{})
if d.get("M1") and d.get("M2"):
    rows.append(["Détection mAP@50","M1 %.3f / M2 %.3f"%(d['M1']['map50'],d['M2']['map50'])])
t=RESULTS.get("tracking",{})
if t: rows.append(["Tracking — interpolées","+%d détections"%t.get("interpolated",0)])
te=RESULTS.get("teams",{})
if te: rows.append(["Équipes — silhouette","%.3f (stabilité 100%% après vote)"%te.get("silhouette",float('nan'))])
h=RESULTS.get("homography",{})
if h: rows.append(["Homographie — correction dyn.","%.1f px moy. (%d coupures)"%(h.get('mean_correction_px',0),len(h.get('shot_cuts',[])))])
pp=RESULTS.get("tactics",{}).get("ppda",{})
if pp: rows.append(["PPDA v1→v2","%d frames ➜ %d événements"%(pp.get('v1_actions_per_frame',0),pp.get('v2',{}).get('pressing_events',0))])
sb_=RESULTS.get("statsbomb",{})
if sb_.get("calibration"): rows.append(["Calibration StatsBomb","rayon=%.1f m"%sb_['calibration']['best_radius']])
print(pd.DataFrame(rows, columns=["Sprint / métrique","Résultat"]).to_string(index=False) if rows else "Aucun résultat (vérifie les inputs).")